# GPU ACCELERATION

In [1]:
%load_ext cudf.pandas
%load_ext cuml.accel

# IMPORTS

In [6]:
import os
import pandas as pd
from playwright.async_api import async_playwright

# MAIN DATASET

In [3]:
df = pd.read_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv')

# URLs TO SCRAPE

In [4]:
evo3_url = 'https://pokemon.fandom.com/wiki/List_of_Pok%C3%A9mon_by_three-stage_evolution'
evo2_url = 'https://pokemon.fandom.com/wiki/List_of_Pok%C3%A9mon_by_two-stage_evolution'

# IMPLEMENT PLAYWRIGHT SCRAPING AND DATA CLEANING

## SCRAPING

In [8]:
DATA_DIR = os.path.expanduser('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/')


async def get_pokes_rows_pw(page, url):
    """Navigate to url and return all non-header rows from .prettytable tables."""
    await page.goto(url, wait_until='domcontentloaded')
    await page.wait_for_selector('.mw-parser-output .prettytable')
    rows = []
    for table in await page.locator('.mw-parser-output .prettytable').all():
        for i, row in enumerate(await table.locator('tr').all()):
            if i > 0:
                rows.append(row)
    return rows


async def get_name(cell):
    """Return the Pokémon name from a table cell (second anchor if present, else first, else cell text)."""
    anchors = await cell.locator('a').all()
    if len(anchors) >= 2:
        return await anchors[1].inner_text()
    elif len(anchors) == 1:
        return await anchors[0].inner_text()
    return await cell.inner_text()


async def pokes_parser1_pw(row):
    """Parse a three-stage evolution table row → dict with ev1/ev2/ev3 keys."""
    cells = await row.locator('td').all()
    n = len(cells)
    if n == 3:
        return {'ev1': await get_name(cells[0]), 'ev2': await get_name(cells[1]), 'ev3': await get_name(cells[2])}
    elif n == 2:
        return {'ev2': await get_name(cells[0]), 'ev3': await get_name(cells[1])}
    else:
        return {'ev3': await get_name(cells[0])}


async def pokes_parser2_pw(row):
    """Parse a two-stage evolution table row → dict with ev1/ev2 keys."""
    cells = await row.locator('td').all()
    n = len(cells)
    if n == 2:
        first_anchors = await cells[0].locator('a').all()
        ev1 = await first_anchors[1].inner_text() if len(first_anchors) >= 2 else await cells[0].inner_text()
        ev2 = await get_name(cells[1])
        return {'ev1': ev1, 'ev2': ev2}
    else:
        cell_anchors = await cells[0].locator('a').all()
        ev2 = await cell_anchors[1].inner_text() if len(cell_anchors) >= 2 else await cells[0].inner_text()
        return {'ev2': ev2}


async def main():
    async with async_playwright() as pw:
        browser = await pw.firefox.launch(headless=True)
        page = await browser.new_page()

        rows3 = await get_pokes_rows_pw(page, evo3_url)
        evo3_data = [await pokes_parser1_pw(r) for r in rows3]
        pd.DataFrame(evo3_data).to_csv(DATA_DIR + 'evo3.csv', index=False)

        rows2 = await get_pokes_rows_pw(page, evo2_url)
        evo2_data = [await pokes_parser2_pw(r) for r in rows2]
        pd.DataFrame(evo2_data).to_csv(DATA_DIR + 'evo2.csv', index=False)

        await browser.close()

await main()
print("Scraping complete. CSVs saved to data directory.")

Scraping complete. CSVs saved to data directory.


## CLEANING

In [9]:
evo3 = pd.read_csv(DATA_DIR + 'evo3.csv')
evo2 = pd.read_csv(DATA_DIR + 'evo2.csv')


# Merge three-stage evolution data into df
df['ev1'] = df['Pokemon'].isin(evo3['ev1'])
df['ev2'] = df['Pokemon'].isin(evo3['ev2'])
df['ev3'] = df['Pokemon'].isin(evo3['ev3'])

df.loc[df['ev1'] == True, ['Evolution Stage', 'Evolves']] = [1, True]
df.loc[df['ev2'] == True, ['Evolution Stage', 'Evolves']] = [2, True]
df.loc[df['ev3'] == True, ['Evolution Stage', 'Evolves']] = [3, False]

df.drop(['ev1', 'ev2', 'ev3'], axis=1, inplace=True)

# Merge two-stage evolution data into df
df['ev1'] = df['Pokemon'].isin(evo2['ev1'])
df['ev2'] = df['Pokemon'].isin(evo2['ev2'])

df.loc[df['ev1'] == True, ['Evolution Stage', 'Evolves']] = [1, True]
df.loc[df['ev2'] == True, ['Evolution Stage', 'Evolves']] = [2, False]

df.drop(['ev1', 'ev2'], axis=1, inplace=True)

# Fill remaining NaN evolution stages (non-evolving Pokémon)
df.loc[df['Evolution Stage'].isna(), ['Evolution Stage', 'Evolves']] = [0, False]

print("Cleaning and merging complete.")

Cleaning and merging complete.


# RENAMING DATAFRAME FOR THE FUN OF IT AND SAVING NEW DATA TO THE `pokemon_dataset_MASTER.csv` FILE FOR SAFE KEEPING

In [11]:
pokes = df

In [13]:
pokes.to_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv', index=False)

# CREATE COLUMNS OF RATIOS BETWEEN THE DIFFERENT STATS

In [14]:
pokes['Attack/Defense'] = pokes['Attack'] / pokes['Defense']
pokes['Special Attack/Special Defense'] = pokes['Special Attack'] / pokes['Special Defense']
pokes['Attack/Special Attack'] = pokes['Attack'] / pokes['Special Attack']
pokes['Defense/Special Defense'] = pokes['Defense'] / pokes['Special Defense']
pokes['Speed/Defense'] = pokes['Speed'] / pokes['Defense']
pokes['Speed/Attack'] = pokes['Speed'] / pokes['Attack']
pokes['HP/Defense'] = pokes['HP'] / pokes['Defense']
pokes['HP/Special Defense'] = pokes['HP'] / pokes['Special Defense']

# SAVE TO THE `,csv` FILE AGAIN (FREQUENT SAVING ENSURES WE DON'T SCREWUP BEYOND FIXING---in my stupid opinion anyhow

In [15]:
pokes.to_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv', index=False)

# LOAD LEGENDARY POKEMON LIST (`legendary_pokes.csv`)

In [17]:
legendary_df = pd.read_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/legendary_pokes.csv')

# CREATE LEGENDARY COLUMN IN MAIN DATASET TO IDENTIFY WHETHER LEGENDARY OR NOT

In [18]:
pokes['Legendary'] = pokes['Pokemon'].isin(legendary_df['Legendary'])

# SAVE IT, FOOOO!

In [20]:
pokes.to_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv', index=False)

# BUILDING OUT THE DATASET FURTHER FOR LATER ANALYSIS

## Phase 0: Data Quality Fixes

In [21]:
STAT_COLS = ['HP', 'Attack', 'Defense', 'Speed', 'Special Attack', 'Special Defense']

# Fix Height and Weight: strip unit strings, cast to float
pokes['Height (m)'] = pd.to_numeric(
    pokes['Height (m)'].astype(str).str.replace(r'[^\d.]', '', regex=True),
    errors='coerce'
)
pokes['Weight (kg)'] = pd.to_numeric(
    pokes['Weight (kg)'].astype(str).str.replace(r'[^\d.]', '', regex=True),
    errors='coerce'
)

# Cast Generation and Evolution Stage to int
pokes['Generation'] = pokes['Generation'].astype(int)
pokes['Evolution Stage'] = pokes['Evolution Stage'].astype(int)

# Fix "Meha" → "Mega" typo in Variation Type
pokes['Variation Type'] = pokes['Variation Type'].replace('Meha', 'Mega')

print("Phase 0: Data quality fixes complete.")
print(f"  Height dtype: {pokes['Height (m)'].dtype}, Weight dtype: {pokes['Weight (kg)'].dtype}")
print(f"  Generation dtype: {pokes['Generation'].dtype}, Evolution Stage dtype: {pokes['Evolution Stage'].dtype}")
print(f"  Variation Types: {sorted(pokes['Variation Type'].dropna().unique())}")

Phase 0: Data quality fixes complete.
  Height dtype: float64, Weight dtype: float64
  Generation dtype: int64, Evolution Stage dtype: int64
  Variation Types: ['Colour', 'Eternamax', 'Forme', 'Gender', 'Mega', 'Mode', 'Partner', 'Regional', 'Regional Forme', 'Regional Mode', 'Size', 'Trainer']


## Phase 1: Derived Analytical Features

In [22]:
# BMI
pokes['BMI'] = pokes['Weight (kg)'] / (pokes['Height (m)'] ** 2)

# Dual Type
pokes['Dual Type'] = pokes['Type 2'].notna()
pokes['Type Count'] = pokes['Dual Type'].astype(int) + 1

# Stat Highest / Lowest
pokes['Stat Highest'] = pokes[STAT_COLS].idxmax(axis=1)
pokes['Stat Lowest'] = pokes[STAT_COLS].idxmin(axis=1)

# Stat Std Dev (how specialized vs balanced a Pokémon is)
pokes['Stat Std Dev'] = pokes[STAT_COLS].std(axis=1)

# Offensive / Defensive totals and ratio
pokes['Offensive Total'] = pokes['Attack'] + pokes['Special Attack'] + pokes['Speed']
pokes['Defensive Total'] = pokes['HP'] + pokes['Defense'] + pokes['Special Defense']
pokes['Offensive/Defensive'] = pokes['Offensive Total'] / pokes['Defensive Total']

# Physical / Special ratio
pokes['Physical/Special'] = (pokes['Attack'] + pokes['Defense']) / (pokes['Special Attack'] + pokes['Special Defense'])

print("Phase 1: Derived analytical features complete.")
print(f"  New columns: BMI, Dual Type, Type Count, Stat Highest, Stat Lowest,")
print(f"  Stat Std Dev, Offensive Total, Defensive Total, Offensive/Defensive, Physical/Special")

Phase 1: Derived analytical features complete.
  New columns: BMI, Dual Type, Type Count, Stat Highest, Stat Lowest,
  Stat Std Dev, Offensive Total, Defensive Total, Offensive/Defensive, Physical/Special


## Phase 2: Categorical Enrichment

In [23]:
# Stat Tier
pokes['Stat Tier'] = pd.cut(
    pokes['Stat Total'],
    bins=[0, 300, 450, 600, float('inf')],
    labels=['Low', 'Mid', 'High', 'Very High'],
    right=False
)

# Speed Tier
pokes['Speed Tier'] = pd.cut(
    pokes['Speed'],
    bins=[0, 50, 100, float('inf')],
    labels=['Slow', 'Medium', 'Fast'],
    right=False
)

# Is Variation
pokes['Is Variation'] = pokes['Variation'].notna()


# Role classification
def classify_role(row):
    if row['Stat Total'] < 300:
        return 'Balanced'

    off_def = row['Offensive/Defensive']
    phys_spec = row['Physical/Special']

    if off_def > 1.3:
        if row['Attack'] > row['Special Attack']:
            return 'Physical Sweeper'
        return 'Special Sweeper'

    if off_def < 0.77:
        if row['Defense'] > row['Special Defense']:
            return 'Physical Wall'
        return 'Special Wall'

    stats = {s: row[s] for s in STAT_COLS}
    top3 = sorted(stats, key=stats.get, reverse=True)[:3]
    if 'Attack' in top3 and 'Special Attack' in top3 and off_def > 1.0:
        return 'Mixed Attacker'

    if 0.8 <= phys_spec <= 1.2 and 0.77 <= off_def <= 1.3:
        return 'Balanced'

    return 'Tank'


pokes['Role'] = pokes.apply(classify_role, axis=1)

print("Phase 2: Categorical enrichment complete.")
print(f"\nStat Tier distribution:\n{pokes['Stat Tier'].value_counts().to_string()}")
print(f"\nSpeed Tier distribution:\n{pokes['Speed Tier'].value_counts().to_string()}")
print(f"\nRole distribution:\n{pokes['Role'].value_counts().to_string()}")

Phase 2: Categorical enrichment complete.

Stat Tier distribution:
Stat Tier
High         547
Mid          386
Low          154
Very High    144
Name: count, dtype: int64

Speed Tier distribution:
Speed Tier
Medium    664
Slow      329
Fast      238
Name: count, dtype: int64

Role distribution:
Role
Tank                389
Balanced            355
Special Sweeper     124
Mixed Attacker      100
Physical Sweeper     97
Physical Wall        85
Special Wall         81
Name: count, dtype: int64


## Phase 3: Normalized / Scaled Stats

In [24]:
# Percentage of Stat Total (makes Pokémon comparable regardless of power level)
for stat in STAT_COLS:
    col_name = stat.replace(' ', '_') + '_pct'
    pokes[col_name] = pokes[stat] / pokes['Stat Total']

# Z-scores (standard for clustering algorithms)
for stat in STAT_COLS:
    col_name = stat.replace(' ', '_') + '_zscore'
    pokes[col_name] = (pokes[stat] - pokes[stat].mean()) / pokes[stat].std()

print("Phase 3: Normalized/scaled stats complete.")
print(f"  Pct columns: {[s.replace(' ', '_') + '_pct' for s in STAT_COLS]}")
print(f"  Z-score columns: {[s.replace(' ', '_') + '_zscore' for s in STAT_COLS]}")
print(f"\nFinal shape: {pokes.shape}")

Phase 3: Normalized/scaled stats complete.
  Pct columns: ['HP_pct', 'Attack_pct', 'Defense_pct', 'Speed_pct', 'Special_Attack_pct', 'Special_Defense_pct']
  Z-score columns: ['HP_zscore', 'Attack_zscore', 'Defense_zscore', 'Speed_zscore', 'Special_Attack_zscore', 'Special_Defense_zscore']

Final shape: (1231, 55)


## Save Enhanced Dataset

In [25]:
pokes.to_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv', index=False)
print(f"Saved: {pokes.shape[1]} columns, {pokes.shape[0]} rows.")

Saved: 55 columns, 1231 rows.


In [26]:
pokes

,Pokedex Number,Pokemon,Link,HP,Attack,Defense,Speed,Special Attack,Special Defense,Stat Total,...,Defense_pct,Speed_pct,Special_Attack_pct,Special_Defense_pct,HP_zscore,Attack_zscore,Defense_zscore,Speed_zscore,Special_Attack_zscore,Special_Defense_zscore
0,1,Bulbasaur,https://bulbapedia.bulbagarden.net/wiki/Bulbas...,45,49,49,45,65,65,318,...,0.154088,0.141509,0.204403,0.204403,-0.977924,-1.017049,-0.866800,-0.832858,-0.282717,-0.302617
1,2,Ivysaur,https://bulbapedia.bulbagarden.net/wiki/Ivysau...,60,62,63,60,80,80,405,...,0.155556,0.148148,0.197531,0.197531,-0.425390,-0.619932,-0.417659,-0.342121,0.158379,0.230318
2,3,Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusa...,80,82,83,80,100,100,525,...,0.158095,0.152381,0.190476,0.190476,0.311322,-0.008983,0.223971,0.312196,0.746506,0.940896
3,3,Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusa...,80,100,123,80,122,120,625,...,0.196800,0.128000,0.195200,0.192000,0.311322,0.540871,1.507231,0.312196,1.393445,1.651475
4,4,Charmander,https://bulbapedia.bulbagarden.net/wiki/Charma...,39,52,43,65,60,50,309,...,0.139159,0.210356,0.194175,0.161812,-1.198938,-0.925407,-1.059289,-0.178542,-0.429748,-0.835551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1226,1023,Iron Crown,https://bulbapedia.bulbagarden.net/wiki/Iron_C...,90,72,100,98,122,108,590,...,0.169492,0.166102,0.206780,0.183051,0.679678,-0.314458,0.769357,0.901080,1.393445,1.225128
1227,1024,Terapagos,https://bulbapedia.bulbagarden.net/wiki/Terapa...,90,65,85,60,65,85,450,...,0.188889,0.133333,0.144444,0.188889,0.679678,-0.528290,0.288134,-0.342121,-0.282717,0.407962
1228,1024,Terapagos,https://bulbapedia.bulbagarden.net/wiki/Terapa...,95,95,110,85,105,110,600,...,0.183333,0.141667,0.175000,0.183333,0.863857,0.388134,1.090172,0.475775,0.893537,1.296186
1229,1024,Terapagos,https://bulbapedia.bulbagarden.net/wiki/Terapa...,160,105,110,85,130,110,700,...,0.157143,0.121429,0.185714,0.157143,3.258172,0.693609,1.090172,0.475775,1.628696,1.296186


In [1]:
import pandas as pd

pokes = pd.read_csv('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv')

pokes.head(33)

,Pokedex Number,Pokemon,Link,HP,Attack,Defense,Speed,Special Attack,Special Defense,Stat Total,...,Defense_pct,Speed_pct,Special_Attack_pct,Special_Defense_pct,HP_zscore,Attack_zscore,Defense_zscore,Speed_zscore,Special_Attack_zscore,Special_Defense_zscore
0,1,Bulbasaur,https://bulbapedia.bulbagarden.net/wiki/Bulbas...,45,49,49,45,65,65,318,...,0.154088,0.141509,0.204403,0.204403,-0.977924,-1.017049,-0.866800,-0.832858,-0.282717,-0.302617
1,2,Ivysaur,https://bulbapedia.bulbagarden.net/wiki/Ivysau...,60,62,63,60,80,80,405,...,0.155556,0.148148,0.197531,0.197531,-0.425390,-0.619932,-0.417659,-0.342121,0.158379,0.230318
2,3,Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusa...,80,82,83,80,100,100,525,...,0.158095,0.152381,0.190476,0.190476,0.311322,-0.008983,0.223971,0.312196,0.746506,0.940896
3,3,Venusaur,https://bulbapedia.bulbagarden.net/wiki/Venusa...,80,100,123,80,122,120,625,...,0.196800,0.128000,0.195200,0.192000,0.311322,0.540871,1.507231,0.312196,1.393445,1.651475
4,4,Charmander,https://bulbapedia.bulbagarden.net/wiki/Charma...,39,52,43,65,60,50,309,...,0.139159,0.210356,0.194175,0.161812,-1.198938,-0.925407,-1.059289,-0.178542,-0.429748,-0.835551
5,5,Charmeleon,https://bulbapedia.bulbagarden.net/wiki/Charme...,58,64,58,80,80,65,405,...,0.143210,0.197531,0.197531,0.160494,-0.499061,-0.558837,-0.578066,0.312196,0.158379,-0.302617
6,6,Charizard,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,84,78,100,109,85,534,...,0.146067,0.187266,0.204120,0.159176,0.237651,0.052112,0.063564,0.966512,1.011163,0.407962
7,6,Charizard,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,130,111,100,130,85,634,...,0.175079,0.157729,0.205047,0.134069,0.237651,1.457295,1.122253,0.966512,1.628696,0.407962
8,6,Charizard,https://bulbapedia.bulbagarden.net/wiki/Chariz...,78,104,78,100,159,115,634,...,0.123028,0.157729,0.250789,0.181388,0.237651,0.663061,0.063564,0.966512,2.481480,1.473831
9,7,Squirtle,https://bulbapedia.bulbagarden.net/wiki/Squirt...,44,48,65,43,50,64,314,...,0.207006,0.136943,0.159236,0.203822,-1.014760,-1.047597,-0.353496,-0.898290,-0.723812,-0.338145
